# Practice: the solver's grid and time step

A sandbox for `lunar_thermal.solver`. Change the grid and the time step and watch what happens to stability and accuracy.

What's here:

1. The default grid, and why it looks the way it does.
2. A baseline solution for a flat equatorial point.
3. **Space:** refine the grid and watch the error fall (second order).
4. **Time:** change the internal step and watch the error, which barely moves.
5. **Stability:** an *explicit* scheme, defined in this notebook, that blows up once the step passes its limit, next to the implicit solver at a step 20 times larger.
6. **Spin-up:** how many lunations it takes to reach periodic steady state, with and without extrapolation.

This notebook only reads from the project. It imports the solver, defines any modified functions itself, and writes nothing to `src/` or to project data. The forcing is a synthetic clipped-cosine Sun (the project's real forcing comes later).

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np

from lunar_thermal import solver as S
from lunar_thermal.properties import (
    RegolithParams,
    conductivity,
    density,
    heat_capacity,
    skin_depth,
)

PARAMS = RegolithParams()  # Hayne et al. (2017) Table A1 defaults
SYNODIC_MONTH = 2.55024e6  # s (Table A1)

## A synthetic forcing

Flat ground on the equator at equinox: sunlight `S cos(h)` while the Sun is up, nothing at night, with Hayne et al.'s angle-dependent albedo (Eq. A8). Time 0 is local noon. Change `N_SAMPLES` to sample the month more or less finely.

In [ ]:
def equatorial_flux(n_samples, period=SYNODIC_MONTH, solar_constant=1361.0):
    # Absorbed flux (W m^-2) at n_samples uniform times over one lunar day.
    dt = period / n_samples
    h = 2 * np.pi * np.arange(n_samples) / n_samples  # hour angle, 0 = noon
    cos_i = np.cos(h)
    theta = np.arccos(np.clip(cos_i, 0, 1))
    albedo = (
        0.12 + 0.06 * (theta / (np.pi / 4)) ** 3 + 0.25 * (theta / (np.pi / 2)) ** 8
    )
    return np.where(cos_i > 0, (1 - albedo) * solar_constant * cos_i, 0.0), dt


N_SAMPLES = 708  # hourly
flux, dt = equatorial_flux(N_SAMPLES)
hours = np.arange(N_SAMPLES) * dt / 3600
plt.figure(figsize=(8, 2.5))
plt.plot(hours / 24, flux)
plt.xlabel("Earth days since local noon")
plt.ylabel("absorbed flux (W m$^{-2}$)")
plt.title(f"Synthetic forcing, dt = {dt:.0f} s")
plt.show()

## 1. The default grid

The first layer is a twentieth of the surface skin depth thick; each layer is 10% thicker than the one above; the column stops at 10 skin depths of the *deep* material, which is more diffusive than the surface material (Hayne et al. Eqs. A31–A33). The result is a few dozen nodes, crowded near the surface where the temperature changes fastest.

In [ ]:
grid_default = S.GridSettings()
z = S.make_grid(PARAMS, N_SAMPLES * dt, grid_default)
zs_top = float(skin_depth(0.0, 250.0, N_SAMPLES * dt, PARAMS))
zs_deep = float(skin_depth(3.0, 250.0, N_SAMPLES * dt, PARAMS))
print(f"{z.size} nodes, top layer {1000 * z[1]:.2f} mm, bottom {z[-1]:.3f} m")
print(
    f"skin depth at 250 K: surface {100 * zs_top:.2f} cm, deep {100 * zs_deep:.2f} cm"
)

fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].plot(np.diff(z) * 1000, z[:-1], "o-", ms=3)
ax[0].invert_yaxis()
ax[0].set_xlabel("layer thickness (mm)")
ax[0].set_ylabel("depth (m)")
ax[1].plot(density(z, PARAMS), z, label="density / 1000")
ax[1].plot(conductivity(z, 250.0, PARAMS) * 1000, z, label="K at 250 K (mW/m/K)")
ax[1].invert_yaxis()
ax[1].legend()
ax[1].set_xlabel("value")
plt.tight_layout()
plt.show()

## 2. Baseline solution

Default settings. The first call compiles the Numba kernels (a few seconds); the compiled code is cached on disk, so later calls are fast.

In [ ]:
t0 = time.perf_counter()
base = S.solve_periodic(flux, dt, output_depths=[0.02, 0.05, 0.1, 0.3])
print(f"{time.perf_counter() - t0:.2f} s (includes compilation on a fresh cache)")
t0 = time.perf_counter()
base = S.solve_periodic(flux, dt, output_depths=[0.02, 0.05, 0.1, 0.3])
print(
    f"{time.perf_counter() - t0:.2f} s warm; {base.n_cycles} cycles, converged={base.converged}, "
    f"residual {base.residual:.1e} K, projected drift {base.drift_estimate:.1e} K"
)
Ts = base.surface_temperature
print(
    f"max {Ts.max():.1f} K, min {Ts.min():.1f} K  (Diviner, via Hayne et al. Table A2: 385 K and 95 K)"
)

plt.figure(figsize=(8, 3.5))
plt.plot(hours / 24, Ts, "k", label="surface")
for j, d in enumerate(base.depths):
    plt.plot(hours / 24, base.depth_temperature[:, j], label=f"{100 * d:.0f} cm")
plt.xlabel("Earth days since local noon")
plt.ylabel("T (K)")
plt.legend(ncol=5, fontsize=8)
plt.show()

## 3. Space: refine the grid

Scale `layers_per_skin_depth` (m) and `growth` (n) together; doubling both roughly halves every layer. Error is measured against a much finer reference. For a second-order scheme, halving the spacing should cut the error about fourfold.

*Try:* change only `m` or only `n`, and see which error (surface or depth) each one controls.

In [ ]:
tight = S.SpinupSettings(tolerance=1e-4)
ref = S.solve_periodic(
    flux,
    dt,
    grid=S.GridSettings(layers_per_skin_depth=160, growth=80, max_step=300),
    spinup=tight,
    output_depths=[0.1],
)
rows = []
for scale in [0.25, 0.5, 1, 2, 4]:
    g = S.GridSettings(
        layers_per_skin_depth=20 * scale, growth=10 * scale, max_step=300
    )
    r = S.solve_periodic(flux, dt, grid=g, spinup=tight, output_depths=[0.1])
    err = np.abs(r.surface_temperature - ref.surface_temperature)
    err_10cm = np.abs(r.depth_temperature - ref.depth_temperature).max()
    rows.append((r.grid.size, err.max(), np.sqrt((err**2).mean()), err_10cm))
    print(
        f"m={20 * scale:5.0f} n={10 * scale:4.0f}: {r.grid.size:4d} nodes, surface max err {err.max():.3f} K, "
        f"rms {np.sqrt((err**2).mean()):.4f} K, 10 cm max err {err_10cm:.3f} K"
    )
rows = np.array(rows)
plt.loglog(rows[:, 0], rows[:, 1], "o-", label="surface, max")
plt.loglog(rows[:, 0], rows[:, 3], "s-", label="10 cm, max")
plt.loglog(
    rows[:, 0], rows[0, 1] * (rows[:, 0] / rows[0, 0]) ** -2.0, "k:", label="slope -2"
)
plt.xlabel("nodes")
plt.ylabel("error vs reference (K)")
plt.legend()
plt.show()

## 4. Time: change the internal step

`max_step` caps the internal step; each forcing interval is split into equal substeps. The scheme is implicit (BDF2), so every step size is stable: the only question is accuracy. Most of the error appears at sunrise, when the surface warms by over 200 K in a few hours.

In [ ]:
ref_t = S.solve_periodic(flux, dt, grid=S.GridSettings(max_step=60), spinup=tight)
for step in [300, 900, 1800, 3600, 3 * 3600, 6 * 3600]:
    # max_step above dt just means one step per forcing interval; to go coarser
    # than the forcing, resample the forcing itself.
    stride = max(1, round(step / dt))
    f_coarse = flux[::stride]
    r = S.solve_periodic(
        f_coarse, dt * stride, grid=S.GridSettings(max_step=step), spinup=tight
    )
    err = np.abs(r.surface_temperature - ref_t.surface_temperature[::stride])
    print(
        f"step {step:6.0f} s ({r.substeps} substep(s) of {dt * stride / r.substeps:6.0f} s): "
        f"max err {err.max():.3f} K, rms {np.sqrt((err**2).mean()):.4f} K"
    )

Beyond one-hour steps the error grows mainly because the *forcing* is sampled more coarsely: sunrise falls between samples, and linear interpolation smears it.

## 5. Stability: an explicit scheme next to the implicit one

The paper's own model (Hayne et al. Appendix A2) steps forward explicitly: the new temperature is computed directly from the old one. That is simple and cheap per step, but stable only if the Fourier number `K dt / (rho c dz^2)` stays below 1/2 (their Eq. A19). With a 2 mm surface layer, that limits the step to a few hundred seconds.

Below, a bare-bones explicit scheme, *defined only in this notebook*, solves the constant-property damped-wave problem (fixed sinusoidal surface temperature) on a uniform grid, just below and just above its limit. Above the limit, the shortest wavelength the grid can hold grows by a factor of `|1 - 4 Fo|` every step and swamps the solution.

In [ ]:
def explicit_damped_wave(
    Fo,
    n_nodes=60,
    depth=0.3,
    rho=1500.0,
    c=600.0,
    K=1e-3,
    T0=250.0,
    A=100.0,
    period=SYNODIC_MONTH,
    n_periods=1.0,
):
    # Forward-Euler (FTCS) solution with a fixed sinusoidal surface. Returns
    # (time step, surface-adjacent temperature history, max |T| reached).
    dz = depth / (n_nodes - 1)
    kappa = K / (rho * c)
    step = Fo * dz**2 / kappa
    n_steps = int(n_periods * period / step)
    w = 2 * np.pi / period
    T = np.full(n_nodes, T0)
    history = []
    for k in range(n_steps):
        t = (k + 1) * step
        T[1:-1] = T[1:-1] + Fo * (T[2:] - 2 * T[1:-1] + T[:-2])
        T[0] = T0 + A * np.sin(w * t)
        T[-1] = T[-2]  # insulated bottom
        history.append(T[1])
        if not np.all(np.isfinite(T)) or np.abs(T).max() > 1e6:
            break
    return step, np.array(history), np.abs(T).max()


fig, ax = plt.subplots(1, 2, figsize=(10, 3))
for a, Fo in zip(ax, [0.45, 0.55]):
    step, hist, tmax = explicit_damped_wave(Fo)
    a.plot(hist)
    a.set_title(f"explicit, Fo = {Fo}: step {step:.0f} s, max |T| {tmax:.3g} K")
    a.set_xlabel("step")
    a.set_ylabel("T at first interior node (K)")
plt.tight_layout()
plt.show()

The project's solver is implicit, so its step is limited by accuracy alone. For the real regolith column, here is the explicit limit at noon (where conductivity is highest) next to the step the solver actually uses:

In [ ]:
w_top = 0.5 * (z[1] - z[0])
T_noon = 385.0
rho_c = density(0.0, PARAMS) * heat_capacity(T_noon, PARAMS)
K_noon = conductivity(0.0, T_noon, PARAMS)
# Explicit update of the top half-cell: dT = dt / (rho c w) * K (T1 - T0) / dz, so
# stability needs dt < rho c w dz / K (one neighbour, no factor 2).
explicit_limit = rho_c * w_top * (z[1] - z[0]) / K_noon
print(
    f"explicit limit for the top layer at noon: ~{explicit_limit:.0f} s "
    f"-> {SYNODIC_MONTH / explicit_limit:.0f} steps per lunation"
)
print(
    f"this solver: {S.GridSettings().max_step:.0f} s steps -> {SYNODIC_MONTH / S.GridSettings().max_step:.0f} steps per lunation"
)

## 6. Spin-up

The temperature at depth settles over many lunations, because the deep column exchanges heat with the surface only through the insulating top few centimetres. Once fast transients die, each lunation shrinks the remaining error by a nearly constant ratio (about 0.95), so the solver can extrapolate to the end of that decay. Compare the cycles needed, and how far each run lands from a tightly converged reference, for several starting temperatures.

*Try:* make `H` larger (a thicker insulating layer) and see how the plain-repetition cycle count changes.

In [ ]:
reference = S.solve_periodic(
    flux, dt, spinup=S.SpinupSettings(tolerance=1e-6, accelerate=False, max_cycles=3000)
)
print(f"reference: {reference.n_cycles} cycles of plain repetition")
for T_start in [None, 150.0, 300.0]:
    for accelerate in [True, False]:
        r = S.solve_periodic(
            flux,
            dt,
            spinup=S.SpinupSettings(
                initial_temperature=T_start, accelerate=accelerate, max_cycles=1000
            ),
        )
        err = np.abs(r.node_temperature - reference.node_temperature).max()
        label = "default" if T_start is None else f"{T_start:.0f} K"
        print(
            f"start {label:>8}, accelerate={accelerate!s:5}: {r.n_cycles:4d} cycles "
            f"({r.extrapolations} extrapolations), max error {err:.4f} K (tolerance {r.tolerance} K)"
        )

### What to notice

- Refining the grid helps both surface and depth error; refining time barely matters at hourly forcing. Spend nodes, not steps.
- The explicit scheme's limit is set by the thinnest layer, which is exactly where accuracy needs thin layers. That conflict is why this solver is implicit.
- A small change from one cycle to the next isn't the same as being close to steady state when the decay ratio is near 1. The solver's convergence test projects the remaining drift for that reason.